In [139]:
#r "nuget: ScottPlot, 5.0.19"
#r "nuget:SkiaSharp.NativeAssets.Linux.NoDependencies"

Installed Packages ScottPlot, 5.0.19 SkiaSharp.NativeAssets.Linux.NoDependencies, 3.119.0

In [140]:
using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using ScottPlot;

In [141]:
public class DefiniteIntegral
{
    private static int _usingResource = 0;
    private static double _result = 0.0;

    public static double Solve(double a, double b, Func<double, double> function, double step, int threadsnumber)
    {
        _result = 0.0;
        double range = b - a;
        double stepSize = range / threadsnumber;
        using Barrier barrier = new Barrier(threadsnumber + 1);

        Thread[] threads = new Thread[threadsnumber];

        for (int i = 0; i < threadsnumber; i++)
        {
            double threadStart = a + i * stepSize;
            double threadEnd = (i == threadsnumber - 1) ? b : threadStart + stepSize;

            threads[i] = new Thread(() =>
            {
                SolvePartially(threadStart, threadEnd, function, step);
                barrier.SignalAndWait();
            });

            threads[i].Start();
        }

        barrier.SignalAndWait();
        
        foreach (var thread in threads)
        {
            thread.Join();
        }

        return _result;
    }

    private static void SolvePartially(double a, double b, Func<double, double> function, double step)
    {
        double current = 0.0;
        double next = 0.0;
        double nextVal = 0.0;

        for (double x = a; x < b; x += step)
        {
            next = Math.Min(x + step, b);
            nextVal = function(next);

            current += (function(x) + nextVal) * (next - x) / 2.0;
        }

        while (!IncrementResult(current))
        {
            Thread.Sleep(1);
        }
    }

    static bool IncrementResult(double current)
    {
        if (0 == Interlocked.Exchange(ref _usingResource, 1))
        {
            _result += current;
        
            Interlocked.Exchange(ref _usingResource, 0);
            return true;
        }

        return false;
    }
}

In [142]:
double SingleThreadIntegral(double a, double b, Func<double, double> function, double step)
{
    double result = 0.0;
    for (double x = a; x < b; x += step)
    {
        double next = Math.Min(x + step, b);
        result += (function(x) + function(next)) * (next - x) / 2.0;
    }
    return result;
}

In [143]:
int a = -100;
int b = 100;
double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
double accuracy = 1e-4;
double bestStep = steps[0];
double bestTime = double.MaxValue;
int iterations = 3;

foreach (var step in steps)
{
    double result = 0;
    Stopwatch sw = Stopwatch.StartNew();
    for (int i = 0; i < iterations; i++)
    {
        result = SingleThreadIntegral(a, b, Math.Sin, step);
    }
    sw.Stop();

    double avgTime = sw.Elapsed.TotalMilliseconds / iterations;

    if (Math.Abs(result) <= accuracy && avgTime < bestTime)
    {
        bestStep = step;
        bestTime = avgTime;
    }
}
display($"Оптимальный шаг: {bestStep}, среднее время: {bestTime:F2} мс");

Оптимальный шаг: 0.1, среднее время: 0.43 мс

In [144]:
bestStep = 1e-1;

int[] threadCounts = { 2, 4, 6, 8, 10, 12, 14, 16 };
double[] avgTimes = new double[threadCounts.Length];

for (int i = 0; i < threadCounts.Length; i++)
{
    double sumTime = 0;
    for (int j = 0; j < iterations; j++)
    {
        var sw = Stopwatch.StartNew();
        DefiniteIntegral.Solve(a, b, Math.Sin, bestStep, threadCounts[i]);
        sw.Stop();
        sumTime += sw.Elapsed.TotalMilliseconds;
    }
    avgTimes[i] = sumTime / repeats;
}

int bestThreads = threadCounts[Array.IndexOf(avgTimes, avgTimes.Min())];
display($"Оптимальное число потоков: {bestThreads}");

Оптимальное число потоков: 4

In [145]:
double singleThreadTime = 0;
for (int j = 0; j < iterations; j++)
{
    var sw = Stopwatch.StartNew();
    SingleThreadIntegral(a, b, Math.Sin, bestStep);
    sw.Stop();
    singleThreadTime += sw.Elapsed.TotalMilliseconds;
}
singleThreadTime /= iterations;

double multiThreadTime = avgTimes[Array.IndexOf(threadCounts, bestThreads)];
double percentDiff = 100.0 * (singleThreadTime - multiThreadTime) / singleThreadTime;

display($"Время однопоточной версии: {singleThreadTime:F2} мс");
display($"Время многопоточной версии: {multiThreadTime:F2} мс");
display($"Разница: {percentDiff:F2}%");

Время однопоточной версии: 0.13 мс

Время многопоточной версии: 0.49 мс

Разница: -290.25%

In [148]:
var scottPlot = new ScottPlot.Plot();

scottPlot.Add.Scatter(avgTimes, threadCounts.Select(x => (double)x).ToArray());

scottPlot

Error: System.TypeInitializationException: The type initializer for 'ScottPlot.Fonts' threw an exception.
 ---> System.TypeInitializationException: The type initializer for 'SkiaSharp.SKTypeface' threw an exception.
 ---> System.DllNotFoundException: Unable to load shared library 'libSkiaSharp' or one of its dependencies. In order to help diagnose loading problems, consider using a tool like strace. If you're using glibc, consider setting the LD_DEBUG environment variable: 
/home/smelson/.nuget/packages/microsoft.dotnet-interactive/1.0.632301/tools/net9.0/any/runtimes/linux-x64/native/libSkiaSharp.so: cannot open shared object file: No such file or directory
/usr/lib/dotnet/shared/Microsoft.NETCore.App/9.0.6/libSkiaSharp.so: cannot open shared object file: No such file or directory
/home/smelson/.nuget/packages/skiasharp/2.88.7/lib/net6.0/libSkiaSharp.so: cannot open shared object file: No such file or directory
/home/smelson/.nuget/packages/microsoft.dotnet-interactive/1.0.632301/tool